# Fase 3 oficial — Perfiles socioeconómicos y cobertura médica

**Grupo 4 · EPHC Paraguay 2024**

**Pregunta reformulada y aprobada:** ¿Qué perfiles sociodemográficos y socioeconómicos se asocian con la falta de seguro médico entre adultos del Paraguay y en qué medida esas características permiten clasificar esa condición?

Esta fase combina clustering exploratorio (sin usar la etiqueta objetivo), clasificación supervisada y NBI departamental como contexto. **No mide riesgo futuro de enfermarse ni establece causalidad.**

## 1. Reproducibilidad
Se parte de los registros anuales originales REG02, se reintegran entrevistas trimestrales para la separación temporal, se conserva la población adulta y se usan factores de expansión para tasas y métricas.

La ejecución crea los resultados si todavía no existen. Las rutas se resuelven desde la raíz del repositorio, incluso si Jupyter se inicia desde `notebooks/`.

In [ ]:
import sys
from pathlib import Path
_raiz=Path.cwd().resolve()
if _raiz.name=='notebooks':
    _raiz=_raiz.parent
sys.path.insert(0,str(_raiz))
from src.fase3 import ejecutar
from IPython.display import display, Image
import json
import pandas as pd
if not (ejecutar.OUT/'resultados.json').exists():
    ejecutar.main()
res=json.loads((ejecutar.OUT/'resultados.json').read_text(encoding='utf-8'))
display(pd.Series({k:res[k] for k in ['n_original','n_adultos','n_desarrollo','n_prueba','k_clusters','silhouette','mejor_modelo','ganador_con_cluster']}))


## 2. Clustering sin la variable de seguro
Los perfiles se construyen con edad, estudio, ingreso, sexo y área; la cobertura se calcula **después** de formar los clusters. La separación y la estabilidad importan tanto como la diferencia descriptiva de porcentajes.

In [ ]:
display(pd.read_csv(ejecutar.OUT/'seleccion_k.csv').round(3))
display(pd.read_csv(ejecutar.OUT/'resumen_perfiles.csv').round(2))
display(Image(filename=str(ejecutar.OUT/'perfiles_cobertura.png')))


## 3. Comparación predictiva
Se compara la clase mayoritaria con regresión logística, Random Forest e HistGradientBoosting. Se usa validación temporal (T1→T2 y T1+T2→T3) para seleccionar la configuración; T4 es el conjunto final de prueba. Los clusters se ajustan únicamente con el entrenamiento de cada división.

In [ ]:
display(pd.read_csv(ejecutar.OUT/'ranking_validacion.csv').round(4))
display(pd.read_csv(ejecutar.OUT/'comparacion_test.csv').query("conjunto=='prueba_T4'").round(3))
display(Image(filename=str(ejecutar.OUT/'matriz_confusion.png')))


## 4. NBI como contexto territorial
Se cruzan indicadores de hogares del Censo 2022 con la cobertura de personas adultas EPHC 2024 a nivel departamental. No puede atribuirse a una persona el NBI de todo su departamento. Las cinco asociaciones son exploratorias.

In [ ]:
display(pd.read_csv(ejecutar.OUT/'correlaciones_nbi.csv').round(3))
display(Image(filename=str(ejecutar.OUT/'nbi_cobertura.png')))


## 5. Conclusión
**El nuevo objetivo de ausencia de cobertura es considerablemente más predecible que la condición de enfermedad reciente** en los respectivos experimentos (no son comparaciones de la misma variable objetivo ni población). La clasificación por Gradient Boosting superó a su baseline con AUC-ROC cercano a 0,793 en T4. Los perfiles evidencian desigualdades descriptivas, pero no mejoraron claramente la predicción y su separación/estabilidad es limitada. Las asociaciones del NBI no superan correcciones por comparaciones múltiples. No interpretar los hallazgos como efectos causales ni criterios para denegar prestaciones.